# Support Vector Machine:

---

## Importing Libraries:

In [1]:
import time

import numpy as np

import pandas as pd

import optuna
from optuna.samplers import TPESampler

from sklearn.model_selection import train_test_split, cross_val_score, cross_validate, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, make_scorer

import warnings
warnings.filterwarnings('ignore')

## Importing Dataset:

In [2]:
data = pd.read_csv("email_spam_features.csv")

## Building Base Model:

In [3]:
X = data.drop(columns=['Category'])
y = data['Category']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=69)

In [4]:
scale = StandardScaler()

X_train = scale.fit_transform(X_train)
X_test = scale.transform(X_test)

In [5]:
label = LabelEncoder()

y_train = label.fit_transform(y_train)
y_test = label.transform(y_test)

In [6]:
svm = SVC()

start_time = time.time()
svm.fit(X_train, y_train)
end_time = time.time() - start_time

print('Model Trained Sucessfully!')
print(f'Time Taken For Training: {end_time:.2f}s')

Model Trained Sucessfully!
Time Taken For Training: 0.19s


In [7]:
y_pred = svm.predict(X_test)

print(f"Accuracy: {(accuracy_score(y_test, y_pred) * 100):.2f}%")

print(f"\nPrecision: {(precision_score(y_test, y_pred)*100):.2f}%")

print(f"\nRecall: {(recall_score(y_test, y_pred)*100):.2f}%")

print(f"\nF1 Score: {(f1_score(y_test, y_pred)*100):.2f}%")

Accuracy: 98.26%

Precision: 99.08%

Recall: 86.40%

F1 Score: 92.31%


In [8]:
matrix = confusion_matrix(y_test, y_pred)
matrix_df = pd.DataFrame(matrix, index=['Actual ham', 'Actual spam'], columns=['Predicted ham', 'Predicted spam'])
matrix_df

,Predicted ham,Predicted spam
Actual ham,906,1
Actual spam,17,108


## Hyper-Parameter Tuning:

In [9]:
def objective(trial):

    kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly'])

    params = {'C' : trial.suggest_categorical('C', [0.01, 0.1, 1, 10, 100]),
             'class_weight' : trial.suggest_categorical('class_weight', ['balanced', None])}
    
    if kernel in ['rbf', 'poly']:
        params['gamma'] = trial.suggest_categorical('gamma', ['scale', 'auto'])

    if (kernel == 'poly'):
        params['degree'] = trial.suggest_int('degree', 2, 5)
    

    model = SVC(kernel=kernel, **params)

    score = np.mean(cross_val_score(estimator=model, X=X_train, y=y_train, scoring='precision', n_jobs=-1, cv=3))
    return score

study = optuna.create_study(direction='maximize', sampler=TPESampler())
study.optimize(objective, n_trials=10, n_jobs=-1)

print("\nBest Parameters:", study.best_params)

[I 2026-10-26 23:00:35,816] A new study created in memory with name: no-name-6e756850-ec4d-4481-bd09-28876b20915c
[I 2026-10-26 23:00:45,013] Trial 3 finished with value: 0.9223699350620911 and parameters: {'kernel': 'rbf', 'C': 100, 'class_weight': 'balanced', 'gamma': 'scale'}. Best is trial 3 with value: 0.9223699350620911.
[I 2026-10-26 23:00:45,115] Trial 0 finished with value: 0.9225159737623466 and parameters: {'kernel': 'rbf', 'C': 100, 'class_weight': 'balanced', 'gamma': 'auto'}. Best is trial 0 with value: 0.9225159737623466.
[I 2026-10-26 23:00:45,523] Trial 2 finished with value: 0.9266388702324636 and parameters: {'kernel': 'rbf', 'C': 10, 'class_weight': 'balanced', 'gamma': 'auto'}. Best is trial 2 with value: 0.9266388702324636.
[I 2026-10-26 23:00:45,560] Trial 1 finished with value: 0.9411265910818823 and parameters: {'kernel': 'poly', 'C': 0.01, 'class_weight': 'balanced', 'gamma': 'scale', 'degree': 3}. Best is trial 1 with value: 0.9411265910818823.
[I 2026-10-26 


Best Parameters: {'kernel': 'linear', 'C': 1, 'class_weight': None}


In [10]:
def objective(trial):

    kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly'])

    params = {'C' : trial.suggest_categorical('C', [0.01, 0.1, 1, 10, 100]),
             'class_weight' : trial.suggest_categorical('class_weight', ['balanced', None])}
    
    if kernel in ['rbf', 'poly']:
        params['gamma'] = trial.suggest_categorical('gamma', ['scale', 'auto'])

    if (kernel == 'poly'):
        params['degree'] = trial.suggest_int('degree', 2, 5)
    

    model = SVC(kernel=kernel, **params)

    score = np.mean(cross_val_score(estimator=model, X=X_train, y=y_train, scoring='f1', n_jobs=-1, cv=3))
    return score

study = optuna.create_study(direction='maximize', sampler=TPESampler())
study.optimize(objective, n_trials=10, n_jobs=-1)

print("\nBest Parameters:", study.best_params)

[I 2026-10-26 23:00:46,767] A new study created in memory with name: no-name-86b76232-b445-417e-9be9-fe04605304ff
[I 2026-10-26 23:00:47,211] Trial 0 finished with value: 0.921722175999462 and parameters: {'kernel': 'poly', 'C': 10, 'class_weight': 'balanced', 'gamma': 'auto', 'degree': 2}. Best is trial 0 with value: 0.921722175999462.
[I 2026-10-26 23:00:47,475] Trial 1 finished with value: 0.9384381511950878 and parameters: {'kernel': 'rbf', 'C': 1, 'class_weight': None, 'gamma': 'scale'}. Best is trial 1 with value: 0.9384381511950878.
[I 2026-10-26 23:00:47,961] Trial 2 finished with value: 0.8838820148671114 and parameters: {'kernel': 'poly', 'C': 0.01, 'class_weight': None, 'gamma': 'scale', 'degree': 5}. Best is trial 1 with value: 0.9384381511950878.
[I 2026-10-26 23:00:48,717] Trial 4 finished with value: 0.9197593498773474 and parameters: {'kernel': 'linear', 'C': 0.01, 'class_weight': 'balanced'}. Best is trial 1 with value: 0.9384381511950878.
[I 2026-10-26 23:00:49,542] T


Best Parameters: {'kernel': 'linear', 'C': 100, 'class_weight': None}


In [11]:
# Priority -> Precision:

Svm = SVC(kernel='linear', C=1, class_weight=None)

start_time = time.time()
Svm.fit(X_train, y_train)
end_time = time.time() - start_time

print('Model Trained Sucessfully!')
print(f'Time Taken For Training: {end_time:.2f}s')

Model Trained Sucessfully!
Time Taken For Training: 0.14s


In [12]:
y_pred = Svm.predict(X_test)

print(f"Accuracy: {(accuracy_score(y_test, y_pred) * 100):.2f}%")

print(f"\nPrecision: {(precision_score(y_test, y_pred)*100):.2f}%")

print(f"\nRecall: {(recall_score(y_test, y_pred)*100):.2f}%")

print(f"\nF1 Score: {(f1_score(y_test, y_pred)*100):.2f}%")

Accuracy: 98.26%

Precision: 97.35%

Recall: 88.00%

F1 Score: 92.44%


In [13]:
matrix = confusion_matrix(y_test, y_pred)
matrix_df = pd.DataFrame(matrix, index=['Actual ham', 'Actual spam'], columns=['Predicted ham', 'Predicted spam'])
matrix_df

,Predicted ham,Predicted spam
Actual ham,904,3
Actual spam,15,110


In [14]:
# Priority -> F1:

SVM = SVC(kernel='linear', C=100, class_weight=None)

start_time = time.time()
SVM.fit(X_train, y_train)
end_time = time.time() - start_time

print('Model Trained Sucessfully!')
print(f'Time Taken For Training: {end_time:.2f}s')

Model Trained Sucessfully!
Time Taken For Training: 2.48s


In [15]:
y_pred = SVM.predict(X_test)

print(f"Accuracy: {(accuracy_score(y_test, y_pred) * 100):.2f}%")

print(f"\nPrecision: {(precision_score(y_test, y_pred)*100):.2f}%")

print(f"\nRecall: {(recall_score(y_test, y_pred)*100):.2f}%")

print(f"\nF1 Score: {(f1_score(y_test, y_pred)*100):.2f}%")

Accuracy: 98.26%

Precision: 97.35%

Recall: 88.00%

F1 Score: 92.44%


In [16]:
matrix = confusion_matrix(y_test, y_pred)
matrix_df = pd.DataFrame(matrix, index=['Actual ham', 'Actual spam'], columns=['Predicted ham', 'Predicted spam'])
matrix_df

,Predicted ham,Predicted spam
Actual ham,904,3
Actual spam,15,110


## Cross-Validation:

In [17]:
# Priority -> Precision:

preprocessing = ColumnTransformer(transformers=[
    ('scale', StandardScaler(), X.columns)
], remainder='passthrough')

model_pipeline = Pipeline(steps=[
     ('preprocessing', preprocessing),
     ('model', SVC(kernel='linear', C=1, class_weight=None))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=69)

scoring = {
    'accuracy' : 'accuracy',
    'precision' : make_scorer(precision_score, pos_label='spam'),
    'recall' : make_scorer(recall_score, pos_label='spam'),
    'f1' : make_scorer(f1_score, pos_label='spam')
}

result = cross_validate(
    estimator=model_pipeline,
    X=X,
    y=y,
    scoring=scoring,
    cv=cv,
    n_jobs=-1
)

metrics = ['accuracy', 'precision', 'recall', 'f1']

for i in metrics:
    score = result[f'test_{i}']
    print(f"{i.capitalize()}: {np.mean(score)*100:.2f}%\n")

Accuracy: 98.60%

Precision: 97.37%

Recall: 91.26%

F1: 94.20%



In [18]:
# Priority -> Precision:

preprocessing = ColumnTransformer(transformers=[
    ('scale', StandardScaler(), X.columns)
], remainder='passthrough')

model_pipeline = Pipeline(steps=[
     ('preprocessing', preprocessing),
     ('model', SVC(kernel='linear', C=100, class_weight=None))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=69)

scoring = {
    'accuracy' : 'accuracy',
    'precision' : make_scorer(precision_score, pos_label='spam'),
    'recall' : make_scorer(recall_score, pos_label='spam'),
    'f1' : make_scorer(f1_score, pos_label='spam')
}

result = cross_validate(
    estimator=model_pipeline,
    X=X,
    y=y,
    scoring=scoring,
    cv=cv,
    n_jobs=-1
)

metrics = ['accuracy', 'precision', 'recall', 'f1']

for i in metrics:
    score = result[f'test_{i}']
    print(f"{i.capitalize()}: {np.mean(score)*100:.2f}%\n")

Accuracy: 98.62%

Precision: 97.38%

Recall: 91.42%

F1: 94.28%

